# Bài 27: Hiệu chỉnh camera

Bài 26 đã giới thiệu ma trận tham số nội $K$ để mô hình hóa một camera lỗ kim. Cách tiếp cận kinh điển (<a href="../references.html#zhang-2000">Zhang, 2000</a>) cho **hiệu chỉnh camera (camera calibration)** khôi phục $K$ bằng cách cho camera xem một mẫu phẳng đã biết (một bàn cờ) từ nhiều góc khác nhau: trích xuất một phép đồng nhất (Bài 24) giữa mặt phẳng bàn cờ và mỗi ảnh, rồi kết hợp các ràng buộc từ nhiều phép đồng nhất như vậy để ước lượng $K$.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Hiệu chỉnh khôi phục những gì

Hiệu chỉnh camera khôi phục hai thứ riêng biệt:

1. **Tham số nội** $K = \begin{bmatrix}f_x & s & c_x\\0 & f_y & c_y\\0&0&1\end{bmatrix}$: tiêu cự, điểm chính và độ xiên (skew) — phần lỗ kim lý tưởng của mô hình camera được dùng xuyên suốt Bài 28 và 29. Với camera thực, có thể yên tâm giả định độ xiên bằng 0.
2. **Hệ số méo (distortion coefficients)** $(k_1, k_2, p_1, p_2, k_3, \dots)$: thấu kính thực không phải lỗ kim hoàn hảo. Méo hướng tâm (radial distortion) ($k_1, k_2, k_3$) uốn cong các đường thẳng thành đường cong (méo thùng (barrel) hoặc méo gối (pincushion), tùy dấu); méo tiếp tuyến (tangential distortion) ($p_1, p_2$) tính đến việc thấu kính không hoàn toàn song song với cảm biến. Mọi phương trình ở Bài 28 và 29 đều ngầm giả định mô hình camera lỗ kim (tức là méo đã được loại bỏ, điều này đòi hỏi phải hiệu chỉnh).

## Một hệ hiệu chỉnh tổng hợp

Ta tạo các ảnh tổng hợp về một camera nhìn vào một bàn cờ: định nghĩa $K$ và độ méo chuẩn, định nghĩa vị trí 3D các góc của một bàn cờ phẳng, và tạo ra vài "bức ảnh" của nó từ các tư thế khác nhau bằng cách chiếu các góc bằng `cv2.projectPoints` (hàm này áp dụng méo y hệt như một thấu kính thực). Điều này cho ta cùng loại đầu vào mà `cv2.calibrateCamera` mong đợi từ các góc bàn cờ thực được phát hiện &mdash; đồng thời cung cấp dữ liệu chuẩn để đối chiếu.

In [ ]:
rng = np.random.default_rng(0)

cols, rows = 9, 6  # internal corners of a 10x7-square checkerboard
corners_3d = np.zeros((rows * cols, 3), dtype=np.float64)
corners_3d[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)
corners_3d[:, :2] -= corners_3d[:, :2].mean(axis=0)  # center the board on its own origin

K_true = np.array([[800, 0, 320], [0, 800, 240], [0, 0, 1]], dtype=np.float64)
dist_true = np.array([-0.3, 0.1, 0.001, -0.0005, 0.02])  # k1, k2, p1, p2, k3
image_size = (640, 480)

object_points, image_points = [], []
for _ in range(40):
    rvec = rng.uniform(-0.4, 0.4, 3)                                     # a somewhat random tilt
    tvec = np.array([rng.uniform(-0.5, 0.5), rng.uniform(-0.4, 0.4), rng.uniform(9, 13)])
    projected, _ = cv2.projectPoints(corners_3d, rvec, tvec, K_true, dist_true)
    projected = projected.reshape(-1, 2)
    if np.all(projected >= 5) and np.all(projected[:, 0] < image_size[0] - 5) and np.all(projected[:, 1] < image_size[1] - 5):
        object_points.append(corners_3d.astype(np.float32))
        image_points.append(projected.astype(np.float32))

print(f'{len(object_points)} usable synthetic views (out of 40 attempted; some fell outside the frame)')

### Kết xuất ảnh tổng hợp

Để có thứ gì đó mà phát hiện góc trong đó, thay vì chỉ có tọa độ điểm, ta cần ảnh thực sự. Vì bàn cờ phẳng, phép ánh xạ từ mẫu bề mặt 2D của chính nó sang một bức ảnh chụp nó chính xác là một phép đồng nhất, nên biến dạng một hình bàn cờ (các ô vuông đen trắng xen kẽ) qua phép đồng nhất của mỗi góc nhìn sẽ tổng hợp ra một bức ảnh &mdash; thay thế cho một bức ảnh chụp thật.

In [ ]:
sq = 60  # pixels per square in the flat target image
board = np.zeros(((rows + 1) * sq, (cols + 1) * sq), dtype=np.uint8)
for r in range(rows + 1):
    for c in range(cols + 1):
        if (r + c) % 2 == 0:
            board[r * sq:(r + 1) * sq, c * sq:(c + 1) * sq] = 255
board_bgr = cv2.cvtColor(board, cv2.COLOR_GRAY2BGR)

# the internal corners' locations in the flat target's own pixel coordinates,
# in the same order as corners_3d
board_pts = np.array([[(c + 1) * sq, (r + 1) * sq] for r in range(rows) for c in range(cols)],
                      dtype=np.float32)

photos = []
for img_pts in image_points:
    H, _ = cv2.findHomography(board_pts, img_pts)
    photos.append(cv2.warpPerspective(board_bgr, H, image_size, borderValue=(128, 128, 128)))

### Phát hiện mẫu hiệu chỉnh

Một quy trình hiệu chỉnh thực tế biết bố cục 2D phẳng của chính bàn cờ (kích thước ô vuông vật lý, do đó có lưới phẳng của `corners_3d`) &mdash; nhưng không biết tư thế của bàn cờ trong không gian 3D so với camera ở mỗi lần chụp (phép xoay và tịnh tiến của nó), và không biết vị trí điểm ảnh 2D nơi các góc của nó rơi vào trong mỗi bức ảnh. Vì vậy bước đầu tiên trong mọi quy trình hiệu chỉnh là tìm các vị trí 2D đó: ta chạy `cv2.findChessboardCorners` của OpenCV trên mỗi bức ảnh &mdash; y như với một bức ảnh thật &mdash; và so sánh các phát hiện của nó với `image_points` chuẩn mà ta đã biết.

In [ ]:
detected_object_points, detected_image_points = [], []
corner_errors = []
example_photo = example_corners_raw = None

for obj_pts, img_pts, photo in zip(object_points, image_points, photos):
    gray = cv2.cvtColor(photo, cv2.COLOR_BGR2GRAY)

    found, corners = cv2.findChessboardCorners(gray, (cols, rows))
    if not found:
        continue
    corners = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1),
                                (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001))

    if example_photo is None:
        example_photo, example_corners_raw = photo, corners

    corners = corners.reshape(-1, 2)[::-1]  # cv2 scans from the opposite corner vs. our board_pts order
    detected_object_points.append(obj_pts)
    detected_image_points.append(corners.astype(np.float32))
    corner_errors.append(np.linalg.norm(corners - img_pts, axis=1).mean())

print(f'corners detected in {len(detected_image_points)} / {len(image_points)} synthetic photos')
print(f'mean per-view corner error vs. ground truth: {np.mean(corner_errors):.2f} px '
      f'(max {np.max(corner_errors):.2f} px)')

example_annotated = example_photo.copy()
cv2.drawChessboardCorners(example_annotated, (cols, rows), example_corners_raw, True)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(board, cmap='gray')
axes[0].set_title('Flat checkerboard target')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(example_annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title('cv2.findChessboardCorners,\none synthetic photo')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Chạy hiệu chỉnh

Thuật toán hiệu chỉnh của Zhang (`cv2.calibrateCamera`) nhận vị trí các góc đã biết trên bàn cờ và vị trí ảnh 2D *được phát hiện* của chúng qua mọi góc nhìn, rồi giải đồng thời ra $K$, độ méo và tư thế riêng của mỗi góc nhìn.

In [ ]:
rms_error, K_estimated, dist_estimated, rvecs, tvecs = cv2.calibrateCamera(
    detected_object_points, detected_image_points, image_size, None, None)

print(f'RMS reprojection error: {rms_error:.2e} pixels\n')
print('K (true):\n', K_true)
print('K (estimated):\n', np.round(K_estimated, 3))
print()
print('distortion (true):     ', dist_true)
print('distortion (estimated):', np.round(dist_estimated.ravel(), 5))

Khi dùng các phát hiện góc có nhiễu, phép hiệu chỉnh không khôi phục chính xác các tham số chuẩn. Ta đo chất lượng hiệu chỉnh bằng sai số chiếu lại. Đây là lý do trên thực tế bạn nên dùng càng nhiều góc nhìn, từ càng nhiều góc độ đa dạng càng tốt trong phạm vi khả thi: nhiều ràng buộc hơn (và đa dạng hơn) sẽ trung bình hóa nhiễu của bộ phát hiện và ràng buộc phép ước lượng tốt hơn.

## Đường thẳng bị thấu kính bẻ cong

Méo thấu kính khiến đường thẳng trở thành đường cong. Trong tọa độ camera chuẩn hóa (trước khi nhân $K$):

$$x_d = x(1 + k_1r^2 + k_2r^4 + k_3r^6) + 2p_1xy + p_2(r^2+2x^2)$$
$$y_d = y(1 + k_1r^2 + k_2r^4 + k_3r^6) + p_1(r^2+2y^2) + 2p_2xy$$

trong đó $r^2 = x^2+y^2$. Đây chính xác là mô hình mà `cv2.projectPoints` đã áp dụng bên trong ở trên.

In [ ]:
def distort_normalized(xy, dist):
    k1, k2, p1, p2, k3 = dist
    x, y = xy[:, 0], xy[:, 1]
    r2 = x**2 + y**2
    radial = 1 + k1 * r2 + k2 * r2**2 + k3 * r2**3
    xd = x * radial + 2 * p1 * x * y + p2 * (r2 + 2 * x**2)
    yd = y * radial + p1 * (r2 + 2 * y**2) + 2 * p2 * x * y
    return np.stack([xd, yd], axis=1)

def make_grid_lines():
    lines = []
    for xv in np.linspace(-0.38, 0.38, 9):
        ys = np.linspace(-0.28, 0.28, 50)
        lines.append(np.stack([np.full_like(ys, xv), ys], axis=1))
    for yv in np.linspace(-0.28, 0.28, 7):
        xs = np.linspace(-0.38, 0.38, 50)
        lines.append(np.stack([xs, np.full_like(xs, yv)], axis=1))
    return lines

def draw_lines(lines_px, height=480, width=640):
    img = np.full((height, width, 3), 255, dtype=np.uint8)
    for line in lines_px:
        for i in range(len(line) - 1):
            cv2.line(img, tuple(line[i].astype(int)), tuple(line[i + 1].astype(int)), (0, 0, 0), 1)
    return img

grid_lines_normalized = make_grid_lines()

straight_px = [line @ K_true[:2, :2].T + K_true[:2, 2] for line in grid_lines_normalized]
distorted_px = [distort_normalized(line, dist_true) @ K_true[:2, :2].T + K_true[:2, 2]
                for line in grid_lines_normalized]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(draw_lines(straight_px))
axes[0].set_title('Ideal pinhole (no distortion)')
axes[1].imshow(draw_lines(distorted_px))
axes[1].set_title(f'Through the lens\n(k1={dist_true[0]}, k2={dist_true[1]})')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Ảnh động này cho thấy độ méo tăng dần liên tục (k1, k2 tăng từ 0 đến giá trị thực, uốn cong lưới), rồi cho thấy bước khử méo kéo lưới bị cong trở lại thẳng bằng phép hiệu chỉnh *ước lượng được* &mdash; cùng hai kết quả đã vẽ ở trên, giờ là một chuyển tiếp liên tục duy nhất.

<img src="../animations/lens_distortion_demo.gif" alt="Ảnh động méo thấu kính: một lưới thẳng bị uốn cong khi các hệ số méo hướng tâm tăng dần, rồi thẳng trở lại nhờ phép khử méo của hiệu chỉnh ước lượng được" style="display: block; margin: 0 auto; max-width: 680px;">

## Khử méo bằng các tham số ước lượng được

`cv2.undistortPoints` đảo ngược mô hình méo, chuyển tọa độ điểm ảnh bị méo trở về tọa độ chuẩn hóa không méo. Áp dụng nó lên các đường lưới bị cong ở trên, dùng phép hiệu chỉnh *ước lượng được* (không phải thực) của ta, sẽ làm chúng thẳng trở lại.

In [ ]:
undistorted_normalized = [
    cv2.undistortPoints(px.reshape(-1, 1, 2).astype(np.float64), K_estimated, dist_estimated).reshape(-1, 2)
    for px in distorted_px
]
undistorted_px = [line @ K_estimated[:2, :2].T + K_estimated[:2, 2] for line in undistorted_normalized]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(draw_lines(distorted_px))
axes[0].set_title('Distorted (as captured)')
axes[1].imshow(draw_lines(undistorted_px))
axes[1].set_title('Undistorted using estimated K, dist')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

original_normalized = np.concatenate(grid_lines_normalized)
recovered_normalized = np.concatenate(undistorted_normalized)
print(f'max recovery error (normalized coords): {np.abs(original_normalized - recovered_normalized).max():.2e}')

Các đường thẳng trở lại, dù không hoàn hảo &mdash; vẫn còn một độ cong nhẹ, vì bản thân `K_estimated`/`dist_estimated` chỉ đúng gần đúng. Quá trình hiệu chỉnh một camera thực này nên được thực hiện, nếu có thể, trước khi đưa ảnh thật vào bất kỳ bộ công cụ hình học nào ở Bài 28 và 29, vốn đều ngầm giả định một mô hình lỗ kim lý tưởng, không méo. Tuy nhiên, hãy nhớ rằng nhiều quy trình hiện đại thực hiện tự hiệu chỉnh (auto-calibration) mà không cần ảnh bàn cờ, dùng các mô hình camera đơn giản hóa (thường chỉ có một tham số tiêu cự &mdash; bỏ qua tỉ lệ khung hình, điểm chính lệch tâm và méo thấu kính).

### Bài tập

1. Thêm nhiễu điểm ảnh tổng hợp (ví dụ `rng.normal(0, 0.5, corners.shape)`) bên trên nhiễu thực của bộ phát hiện, vào các góc được phát hiện của mỗi góc nhìn trước khi hiệu chỉnh. Sai số chiếu lại RMS tăng bao nhiêu, và `K_estimated` trôi xa hơn khỏi `K_true` như thế nào?
2. Thiết lập này kết thúc với chỉ một nhúm góc nhìn được phát hiện thành công (đã in ở trên). Hãy thử giảm `detected_object_points`/`detected_image_points` xuống chỉ còn 3 góc nhìn trước khi hiệu chỉnh. Ước lượng có suy giảm thấy rõ không, đặc biệt là với các hệ số méo (vốn cần các góc trải rộng khắp khung hình, kể cả gần các cạnh, để được ràng buộc tốt)?
3. Đặt `dist_true = np.zeros(5)` (một lỗ kim hoàn hảo, không méo) và chạy lại toàn bộ notebook. Xác nhận lưới "bị méo" và lưới "lý tưởng" trở nên giống hệt nhau, và phép hiệu chỉnh vẫn khôi phục `K_true` khá tốt ngay cả khi độ méo cần ước lượng bằng 0.